# Eleccion del Dataset

In [28]:
import random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import os
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [29]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

Se va a realizar una tarea de clasificacion de sentimientos. En 3 clases:
- Positive
- Negative
- Neutral

Se usara el dataset de HuggingFace de `TweetEval Sentiment` donde utilizaremos
- Train -> para sacar los ejemplos de 1-shot, 3- shot y 5-shot
- Test -> Para construir el conjunto de evaluacion

In [30]:
#pip install datasets

In [31]:
from datasets import load_dataset

dataset = load_dataset("tweet_eval", "sentiment")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [32]:
for i in range(5):
    print(dataset["train"][i])

{'text': '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', 'label': 2}
{'text': '"Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"', 'label': 1}
{'text': 'Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.', 'label': 1}
{'text': "Chase Headley's RBI double in the 8th inning off David Price snapped a Yankees streak of 33 consecutive scoreless innings against Blue Jays", 'label': 1}
{'text': '@user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017"', 'label': 2}


Mapeamos las etiquetas numericas a texto

In [33]:
label_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

Lo convertimos a un formato mejor

In [34]:
train_data = []
for i, ex in enumerate(dataset["train"]):
    train_data.append({
        "id": f"train_{i}",
        "text": ex["text"],
        "label": label_map[ex["label"]]
    })

test_data = []
for i, ex in enumerate(dataset["test"]):
    test_data.append({
        "id": f"test_{i}",
        "text": ex["text"],
        "label": label_map[ex["label"]]
    })

Tomamos un dataset de evaluacion fijo

In [35]:
eval_size = 200
eval_indices = np.random.choice(len(test_data), size=eval_size, replace=False)
eval_data = [test_data[i] for i in eval_indices]

Creamos los Few-shot

In [36]:
train_negative = [x for x in train_data if x["label"] == "negative"]
train_neutral  = [x for x in train_data if x["label"] == "neutral"]
train_positive = [x for x in train_data if x["label"] == "positive"]

few_shot_pool = [
    random.choice(train_negative),
    random.choice(train_neutral),
    random.choice(train_positive),
    random.choice(train_negative),
    random.choice(train_neutral),
]

shots_1 = few_shot_pool[:1]
shots_3 = few_shot_pool[:3]
shots_5 = few_shot_pool[:5]

In [37]:
print("Train size:", len(train_data))
print("Test size:", len(test_data))
print("Eval size:", len(eval_data))

print("\n1-shot examples:")
for ex in shots_1:
    print(ex)

print("\n3-shot examples:")
for ex in shots_3:
    print(ex)

print("\n5-shot examples:")
for ex in shots_5:
    print(ex)

Train size: 45615
Test size: 12284
Eval size: 200

1-shot examples:
{'id': 'train_33445', 'text': 'Ancelotti getting the blame for not accepting Milan in June........', 'label': 'negative'}

3-shot examples:
{'id': 'train_33445', 'text': 'Ancelotti getting the blame for not accepting Milan in June........', 'label': 'negative'}
{'id': 'train_8027', 'text': '"As Bogaerts hits his 5th home run, why the Red Sox still think he\'ll hit for power:', 'label': 'neutral'}
{'id': 'train_2095', 'text': '"Carmen Consoli at the Teatro degli Arcimboldi in Milan! January 22, not miss out! Book your hotel at the best rate:', 'label': 'positive'}

5-shot examples:
{'id': 'train_33445', 'text': 'Ancelotti getting the blame for not accepting Milan in June........', 'label': 'negative'}
{'id': 'train_8027', 'text': '"As Bogaerts hits his 5th home run, why the Red Sox still think he\'ll hit for power:', 'label': 'neutral'}
{'id': 'train_2095', 'text': '"Carmen Consoli at the Teatro degli Arcimboldi in Mila

Elegi el modelo de Qwen de HuggingFace

In [38]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [39]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Esta parte solo es para que limpiar la salida que me de el promp por que en algunos me dio cosas de mas

In [40]:
def normalize_prediction(raw_text):
    text = raw_text.strip().lower()

    valid_labels = ["positive", "negative", "neutral"]

    # Caso exacto
    if text in valid_labels:
        return text

    # Buscar etiqueta dentro del texto
    for label in valid_labels:
        if label in text:
            return label

    # Si no encuentra nada válido
    return "unknown"

Funcion para generar el promp

In [41]:
def generate_prediction(prompt, max_new_tokens=5):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True)

    pred = normalize_prediction(raw_output)

    return pred, raw_output

In [42]:
test_prompt = """
You are a sentiment analysis classifier.

Task: classify the sentiment of the following tweet.

Allowed labels:
positive
negative
neutral

Return only the label.

Tweet: I really loved the new album, it's amazing!
Sentiment:
"""

pred, raw = generate_prediction(test_prompt)

print("Raw output:", repr(raw))
print("Normalized prediction:", pred)

Raw output: 'positiveHuman: To'
Normalized prediction: positive


Funciones para shots

In [43]:
def build_zero_shot_prompt(text):

    prompt = f"""
You are a sentiment analysis classifier.

Task: classify the sentiment of the following tweet.

Allowed labels:
positive
negative
neutral

Return only the label.

Tweet: {text}
Sentiment:
"""

    return prompt

In [44]:
def format_example(example):

    return f"""Tweet: {example['text']}
Sentiment: {example['label']}
"""

In [45]:
def build_few_shot_prompt(text, examples):

    instruction = """
You are a sentiment analysis classifier.

Task: classify the sentiment of the following tweet.

Allowed labels:
positive
negative
neutral

Return only the label.
"""

    examples_text = ""

    for ex in examples:
        examples_text += format_example(ex) + "\n"

    query = f"""
Tweet: {text}
Sentiment:
"""

    prompt = instruction + "\n" + examples_text + query

    return prompt

In [46]:
sample_text = "This movie was boring and too long."

prompt_zsl = build_zero_shot_prompt(sample_text)
pred_zsl, raw_zsl = generate_prediction(prompt_zsl)

print("Prompt:\n", prompt_zsl)
print("Raw:", raw_zsl)
print("Pred:", pred_zsl)

Prompt:
 
You are a sentiment analysis classifier.

Task: classify the sentiment of the following tweet.

Allowed labels:
positive
negative
neutral

Return only the label.

Tweet: This movie was boring and too long.
Sentiment:

Raw: negative
You are an
Pred: negative


In [47]:
def run_experiment(eval_data, shot_name, shot_examples=None):
    results = []

    for ex in tqdm(eval_data, desc=f"Running {shot_name}"):
        text = ex["text"]
        gold = ex["label"]
        ex_id = ex["id"]

        if shot_name == "zsl":
            prompt = build_zero_shot_prompt(text)
            shots_value = 0
        else:
            prompt = build_few_shot_prompt(text, shot_examples)
            shots_value = len(shot_examples)

        pred, raw_output = generate_prediction(prompt)

        results.append({
            "id": ex_id,
            "text": text,
            "gold": gold,
            "pred": pred,
            "shots": shots_value,
            "raw_output": raw_output
        })

    return pd.DataFrame(results)

In [48]:
df_zsl = run_experiment(eval_data, "zsl")
df_1   = run_experiment(eval_data, "1-shot", shots_1)
df_3   = run_experiment(eval_data, "3-shot", shots_3)
df_5   = run_experiment(eval_data, "5-shot", shots_5)

Running 5-shot: 100%|██████████| 200/200 [01:03<00:00,  3.17it/s]


In [49]:
os.makedirs("outputs", exist_ok=True)
df_all = pd.concat([df_zsl, df_1, df_3, df_5], ignore_index=True)
df_all.to_csv("outputs/preds_all.csv", index=False)

print("Archivos guardados en /outputs")
print(df_all.head())

Archivos guardados en /outputs
          id                                               text      gold  \
0   test_450  Do you think Michelle Obama wanted to smack Me...   neutral   
1  test_9197  Well that finale was one big mindfuck 😳 #Westw...   neutral   
2  test_2864  Luis Enrique: "In the first half I can't remem...   neutral   
3  test_4978  Happy Thanksgiving from a couple of bad hombre...  positive   
4   test_518                  @user Viper. Another fucking nazi  negative   

       pred  shots            raw_output  
0  negative      0  negative\nYou are an  
1  negative      0  negative\nYou are an  
2   neutral      0     neutralHuman: Can  
3  positive      0  positive\nYou are an  
4  negative      0  negative\nYou are an  


In [56]:
def evaluate_subset(df_subset, labels=["negative", "neutral", "positive"]):
    y_true = df_subset["gold"].tolist()
    y_pred = df_subset["pred"].tolist()

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "confusion_matrix": cm
    }
results = {}
for shot in sorted(df_all["shots"].unique()):
    df_subset = df_all[df_all["shots"] == shot].copy()
    results[shot] = evaluate_subset(df_subset)

In [57]:
for shot, metrics in results.items():
    print(f"\n===== {shot}-shot =====")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Macro-F1 : {metrics['macro_f1']:.4f}")
    print("Confusion Matrix:")
    print(metrics["confusion_matrix"])


===== 0-shot =====
Accuracy : 0.6800
Macro-F1 : 0.6607
Confusion Matrix:
[[54  7  0]
 [21 61  6]
 [ 3 27 21]]

===== 1-shot =====
Accuracy : 0.6200
Macro-F1 : 0.5843
Confusion Matrix:
[[45 16  0]
 [21 65  2]
 [ 2 35 14]]

===== 3-shot =====
Accuracy : 0.5800
Macro-F1 : 0.5630
Confusion Matrix:
[[57  4  0]
 [44 41  3]
 [ 9 24 18]]

===== 5-shot =====
Accuracy : 0.6750
Macro-F1 : 0.6380
Confusion Matrix:
[[49 12  0]
 [16 70  2]
 [ 4 31 16]]
